In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

def chol_psd(A: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    """
    Cholesky-like factorization for PSD matrices.
    Returns lower-triangular L such that L @ L.T ≈ A.
    Small negative pivots within [-tol, 0] are treated as zero.
    """
    A = np.array(A, dtype=float, copy=False)
    n = A.shape[0]
    L = np.zeros_like(A)
    for j in range(n):
        # diagonal
        s = 0.0 if j == 0 else float(L[j, :j] @ L[j, :j])
        dj = A[j, j] - s
        if dj < -tol:
            raise np.linalg.LinAlgError(f"Matrix not PSD at pivot {j}: {dj}")
        dj = 0.0 if (-tol <= dj <= 0.0) else dj
        L[j, j] = np.sqrt(dj) if dj > 0.0 else 0.0

        # off-diagonals
        if L[j, j] > 0.0:
            inv = 1.0 / L[j, j]
            for i in range(j + 1, n):
                s = 0.0 if j == 0 else float(L[i, :j] @ L[j, :j])
                L[i, j] = (A[i, j] - s) * inv
        else:
            # column j remains zeros when pivot is (near) zero
            L[j+1:, j] = 0.0
    return L

# Read data and give output
DATA_DIR = Path.cwd() / "testfiles_" / "data"
csv_path = DATA_DIR / "testout_3.1.csv"

M = pd.read_csv(csv_path)
for c in M.columns:
    M[c] = pd.to_numeric(M[c], errors="coerce")

if M.shape[0] != M.shape[1]:
    raise ValueError(f"Matrix must be square, got {M.shape}")
M.index = M.columns

# symmetrize lightly to reduce csv round-off
A = 0.5 * (M.to_numpy(float) + M.to_numpy(float).T)

# compute PSD Cholesky
L = chol_psd(A, tol=1e-10)

L_df = pd.DataFrame(L, index=M.index, columns=M.columns)
print(L_df)

          x1        x2        x3        x4            x5
x1  1.083506  0.000000  0.000000  0.000000  0.000000e+00
x2 -0.570360  0.996437  0.000000  0.000000  0.000000e+00
x3 -0.262628 -0.133175  0.911807  0.000000  0.000000e+00
x4 -0.060130  0.412871  0.431384  0.731160  0.000000e+00
x5 -0.635240 -0.223938  0.054179 -0.256892  1.490116e-08
